# ANN topological simplification pipeline

This notebook trains the requested feedforward models, captures hidden activations on a fixed probe set, runs ripser, computes COM, and plots COM vs architecture size.

Pipeline order:
1. Training
2. Activation capture
3. Ripser
4. COM
5. Plotting


In [ ]:
# Cell 1: setup

# !pip3 -q install ripser dill scipy seaborn pandas scikit-learn

import os
import json
import math
import random
from pathlib import Path

import dill
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

from ripser import ripser
from scipy.stats import pearsonr, spearmanr, kendalltau

# Reproducibility
GLOBAL_SEED = 0
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

# Paths
ROOT = Path('${TDL_ROOT_DIR}/John/ANN')
DATA_PATH = ROOT / 'full_dataset.npz'
MODEL_ROOT = ROOT / 'models'
RIPSER_ROOT = ROOT / 'ripser_results'
FIG_ROOT = ROOT / 'figures'
LOG_ROOT = ROOT / 'logs'
MODEL_SIZES_JSON = ROOT / 'model_sizes.json'

for p in [MODEL_ROOT, RIPSER_ROOT, FIG_ROOT, LOG_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Experiment settings
N_SEEDS = 30
TARGET_ACC = 0.99
BATCH_SIZE = 256
MAX_EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 0.0
PATIENCE = 20

# Ripser / COM settings
ETA = 2.5
DIMS = (0,)
N_PERM = 15   # closest direct ripser analogue to the user's k = 15 request
USE_RUNNING_MIN = True
INCLUDE_OUTPUT = True
DIR_NAME = f'ripser_k{N_PERM}_eta{ETA}'


In [ ]:
# Cell 2: load data and build a fixed probe set

def load_npz_dataset(path: Path):
    data = np.load(path, allow_pickle=True)
    keys = set(data.files)

    # Common conventions
    if {'X_train', 'y_train', 'X_test', 'y_test'} <= keys:
        X_train = data['X_train']
        y_train = data['y_train']
        X_test = data['X_test']
        y_test = data['y_test']
        return (X_train, y_train), (X_test, y_test)

    if {'X', 'y'} <= keys:
        X = data['X']
        y = data['y']
        return (X, y), None

    # Fallback: try to infer the two largest arrays as X and y
    arrays = [(k, data[k]) for k in data.files if isinstance(data[k], np.ndarray)]
    if len(arrays) < 2:
        raise ValueError(f'Could not infer dataset arrays from {path}; found keys: {data.files}')

    arrays = sorted(arrays, key=lambda kv: kv[1].size, reverse=True)
    X = arrays[0][1]
    y = arrays[1][1]
    return (X, y), None

(dataset_train, dataset_test) = load_npz_dataset(DATA_PATH)
X_all, y_all = dataset_train

X_all = np.asarray(X_all)
y_all = np.asarray(y_all)

# Convert labels to integer class indices if needed
if y_all.ndim > 1:
    y_all = y_all.argmax(axis=-1)
y_all = y_all.astype(np.int64)

# Basic shape normalization
if X_all.ndim == 2:
    # [N, D] already fine
    pass
elif X_all.ndim == 3:
    # [N, C, L] or similar; flatten to vectors for a vanilla MLP
    X_all = X_all.reshape(X_all.shape[0], -1)
else:
    X_all = X_all.reshape(X_all.shape[0], -1)

print('X shape:', X_all.shape)
print('y shape:', y_all.shape)
print('num classes:', int(np.max(y_all)) + 1)

# Fixed probe set used for activation capture and ripser
# Keep this identical across every model.
PROBE_SIZE = min(1024, len(X_all))
probe_idx = np.random.RandomState(GLOBAL_SEED).choice(len(X_all), size=PROBE_SIZE, replace=False)
X_probe = X_all[probe_idx]
y_probe = y_all[probe_idx]

# Train/val split
N = len(X_all)
N_VAL = int(0.1 * N)
N_TRAIN = N - N_VAL
train_ds_full = TensorDataset(torch.tensor(X_all, dtype=torch.float32), torch.tensor(y_all, dtype=torch.long))
train_ds, val_ds = random_split(
    train_ds_full,
    [N_TRAIN, N_VAL],
    generator=torch.Generator().manual_seed(GLOBAL_SEED),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
probe_loader = DataLoader(TensorDataset(torch.tensor(X_probe, dtype=torch.float32)), batch_size=BATCH_SIZE, shuffle=False)

input_dim = X_all.shape[1]
num_classes = int(np.max(y_all)) + 1
print('input_dim:', input_dim)
print('num_classes:', num_classes)
print('probe size:', PROBE_SIZE)


In [ ]:
# Cell 3: model grid

ARCHS = {
    '30x8': [30, 30, 30, 30, 30, 30, 30, 30],
    '24x8': [24, 24, 24, 24, 24, 24, 24, 24],
    '18x8': [18, 18, 18, 18, 18, 18, 18, 18],
    '30x4_24x4': [30, 30, 30, 30, 24, 24, 24, 24],
    '30x4_18x4': [30, 30, 30, 30, 18, 18, 18, 18],
    '30x4_12x4': [30, 30, 30, 30, 12, 12, 12, 12],
}

ACTIVATIONS = ['relu', 'tanh', 'leaky_relu']

MODEL_SIZES = {name: int(sum(dims)) for name, dims in ARCHS.items()}
with open(MODEL_SIZES_JSON, 'w') as f:
    json.dump(MODEL_SIZES, f, indent=2)

print(MODEL_SIZES)

# Save a small manifest for convenience
manifest = pd.DataFrame([
    {'arch': k, 'hidden_dims': v, 'hidden_sum': sum(v)} for k, v in ARCHS.items()
])
manifest


In [ ]:
# Cell 4: generic feedforward model and training utilities

class FeedForwardNet(nn.Module):
    def __init__(self, input_dim, hidden_dims, num_classes, activation_name='relu'):
        super().__init__()
        act_map = {
            'relu': nn.ReLU,
            'tanh': nn.Tanh,
            'leaky_relu': nn.LeakyReLU,
        }
        if activation_name not in act_map:
            raise ValueError(f'Unknown activation: {activation_name}')
        act_cls = act_map[activation_name]

        layers = []
        dims = [input_dim] + list(hidden_dims)
        for i in range(len(hidden_dims)):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            layers.append(act_cls())
        layers.append(nn.Linear(dims[-1], num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def build_model(hidden_dims, activation_name):
    return FeedForwardNet(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        num_classes=num_classes,
        activation_name=activation_name,
    )


def accuracy_from_logits(logits, y):
    preds = logits.argmax(dim=-1)
    return (preds == y).float().mean().item()


def evaluate(model, loader, device=DEVICE):
    model.eval()
    total_correct = 0
    total = 0
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * len(xb)
            total_correct += (logits.argmax(dim=-1) == yb).sum().item()
            total += len(xb)
    return {'loss': total_loss / max(total, 1), 'acc': total_correct / max(total, 1)}


def train_one_model(hidden_dims, activation_name, seed, save_path, max_epochs=MAX_EPOCHS, target_acc=TARGET_ACC):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = build_model(hidden_dims, activation_name).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = -1.0
    best_state = None
    best_epoch = -1
    patience_left = PATIENCE
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(xb)
            train_correct += (logits.argmax(dim=-1) == yb).sum().item()
            train_total += len(xb)

        train_loss /= max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)
        val_metrics = evaluate(model, val_loader)

        row = {
            'seed': seed,
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['acc'],
        }
        history.append(row)

        if val_metrics['acc'] > best_val_acc:
            best_val_acc = val_metrics['acc']
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1

        if best_val_acc >= target_acc:
            break
        if patience_left <= 0:
            break

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.to('cpu')
    model.eval()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'state_dict': model.state_dict(),
        'hidden_dims': hidden_dims,
        'activation_name': activation_name,
        'seed': seed,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch,
    }, save_path)

    return model, pd.DataFrame(history)


In [ ]:
# Cell 5: train the full grid and save checkpoints

# Expected output layout:
# models/<arch>/<activation>/seed_<seed>.pth

train_summaries = []

for arch_name, hidden_dims in ARCHS.items():
    for act_name in ACTIVATIONS:
        for seed in range(N_SEEDS):
            ckpt_path = MODEL_ROOT / arch_name / act_name / f'seed_{seed}.pth'
            print(f'Training {arch_name} | {act_name} | seed {seed}')
            model, hist = train_one_model(hidden_dims, act_name, seed, ckpt_path)

            final_val = evaluate(model.to(DEVICE), val_loader)
            final_train = evaluate(model.to(DEVICE), train_loader)
            model.to('cpu')

            hist_path = LOG_ROOT / arch_name / act_name / f'seed_{seed}.csv'
            hist_path.parent.mkdir(parents=True, exist_ok=True)
            hist.to_csv(hist_path, index=False)

            train_summaries.append({
                'arch': arch_name,
                'activation': act_name,
                'seed': seed,
                'checkpoint': str(ckpt_path),
                'train_acc': final_train['acc'],
                'val_acc': final_val['acc'],
                'train_loss': final_train['loss'],
                'val_loss': final_val['loss'],
            })

summary_df = pd.DataFrame(train_summaries)
summary_df.head()


In [ ]:
# Cell 6: activation capture helpers

def load_trained_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    hidden_dims = ckpt['hidden_dims']
    activation_name = ckpt['activation_name']
    model = build_model(hidden_dims, activation_name)
    model.load_state_dict(ckpt['state_dict'], strict=True)
    model.eval()
    return model, hidden_dims, activation_name


def get_hidden_linear_names(model):
    # All Linear layers except the final classifier layer
    names = []
    modules = list(model.named_modules())
    linear_names = [name for name, module in modules if isinstance(module, nn.Linear)]
    if len(linear_names) < 2:
        raise ValueError('Model must have at least one hidden Linear layer and one output Linear layer.')
    return linear_names[:-1]


@torch.no_grad()
def capture_hidden_activations(model, loader, layer_names):
    modules = dict(model.named_modules())
    for name in layer_names:
        if name not in modules:
            raise KeyError(f'Layer {name} not found in model. Available names include: {list(modules.keys())[:20]}')

    buffers = {name: [] for name in layer_names}
    handles = []

    def make_hook(layer_name):
        def hook(module, inp, out):
            y = out.detach().cpu()
            if y.ndim > 2:
                y = y.reshape(y.shape[0], -1)
            buffers[layer_name].append(y.numpy())
        return hook

    for name in layer_names:
        handles.append(modules[name].register_forward_hook(make_hook(name)))

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            _ = model.to(DEVICE)(xb)

    for h in handles:
        h.remove()

    activations = {name: np.concatenate(buffers[name], axis=0) for name in layer_names}
    return activations


In [ ]:
# Cell 7: capture activations for every checkpoint

ACT_ROOT = ROOT / 'activations'
ACT_ROOT.mkdir(parents=True, exist_ok=True)

activation_manifest = []

for arch_name in ARCHS:
    for act_name in ACTIVATIONS:
        ckpt_dir = MODEL_ROOT / arch_name / act_name
        if not ckpt_dir.exists():
            print('Skipping missing', ckpt_dir)
            continue

        for seed in range(N_SEEDS):
            ckpt_path = ckpt_dir / f'seed_{seed}.pth'
            if not ckpt_path.exists():
                print('Missing checkpoint:', ckpt_path)
                continue

            model, hidden_dims, activation_name = load_trained_model(ckpt_path)
            layer_names = get_hidden_linear_names(model)
            acts = capture_hidden_activations(model, probe_loader, layer_names)

            out_dir = ACT_ROOT / arch_name / act_name / f'seed_{seed}'
            out_dir.mkdir(parents=True, exist_ok=True)

            # Save the fixed probe input as the baseline 'input layer' point cloud
            with open(out_dir / 'input_layer.pkl', 'wb') as f:
                dill.dump(X_probe, f)

            for lname, A in acts.items():
                safe_name = lname.replace('.', '_')
                with open(out_dir / f'{safe_name}.pkl', 'wb') as f:
                    dill.dump(A, f)

            activation_manifest.append({
                'arch': arch_name,
                'activation': act_name,
                'seed': seed,
                'checkpoint': str(ckpt_path),
                'activation_dir': str(out_dir),
                'layer_names': layer_names,
            })

activation_manifest_df = pd.DataFrame(activation_manifest)
activation_manifest_df.head()


In [ ]:
# Cell 8: ripser utilities

def standardize_features(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    mu = x.mean(axis=0, keepdims=True)
    sd = x.std(axis=0, keepdims=True)
    return (x - mu) / (sd + eps)


def compute_diagrams_for_point_cloud(X, maxdim=1, standardize=True, n_perm=15):
    X = np.asarray(X, dtype=np.float32)
    ok = np.isfinite(X).all(axis=1)
    X = X[ok]
    if X.shape[0] < 3:
        raise ValueError('Need at least 3 points for ripser.')
    if standardize:
        X = standardize_features(X)
    return ripser(X, maxdim=maxdim, n_perm=n_perm)['dgms']


def save_diagrams(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        dill.dump(obj, f)


def load_pickle(path):
    with open(path, 'rb') as f:
        return dill.load(f)


In [ ]:
# Cell 9: run ripser on every captured activation set

RIP_ROOT = RIPSER_ROOT / DIR_NAME
RIP_ROOT.mkdir(parents=True, exist_ok=True)

ripser_manifest = []

for row in activation_manifest:
    arch_name = row['arch']
    act_name = row['activation']
    seed = row['seed']
    act_dir = Path(row['activation_dir'])

    out_dir = RIP_ROOT / arch_name / act_name / f'seed_{seed}'
    out_dir.mkdir(parents=True, exist_ok=True)

    input_cloud = load_pickle(act_dir / 'input_layer.pkl')
    input_dgm = compute_diagrams_for_point_cloud(
        input_cloud,
        maxdim=1,
        standardize=True,
        n_perm=N_PERM,
    )
    save_diagrams(input_dgm, out_dir / 'input_layer.pkl')

    layer_files = sorted([p for p in act_dir.glob('*.pkl') if p.name != 'input_layer.pkl'])
    layer_diagrams = []

    for p in layer_files:
        A = load_pickle(p)
        dgm = compute_diagrams_for_point_cloud(
            A,
            maxdim=1,
            standardize=True,
            n_perm=N_PERM,
        )
        layer_diagrams.append(dgm)

    save_diagrams(layer_diagrams, out_dir / 'model.pkl')

    ripser_manifest.append({
        'arch': arch_name,
        'activation': act_name,
        'seed': seed,
        'ripser_dir': str(out_dir),
        'num_layers': len(layer_diagrams),
    })

ripser_manifest_df = pd.DataFrame(ripser_manifest)
ripser_manifest_df.head()


## COM pipeline  (Cells 10–13)

Cells 10–12 define the ANN-native COM helpers.  
Cell 13 shows three ready-to-run example plots.


In [ ]:
# Cell 10: COM core helpers (feedforward ANN version)

def _betti_number_from_diagrams(dgm_list, dim, eta):
    """Count persistent pairs with lifetime > eta in dimension `dim`."""
    if dim >= len(dgm_list):
        return 0
    dgm = dgm_list[dim]          # shape (n_pairs, 2)
    if dgm is None or len(dgm) == 0:
        return 0
    dgm = np.asarray(dgm, dtype=float)
    finite_mask = np.isfinite(dgm[:, 1])
    lifetimes   = dgm[finite_mask, 1] - dgm[finite_mask, 0]
    return int(np.sum(lifetimes > eta))


def get_betti_curve_for_seed(
    arch_name: str,
    act_name:  str,
    seed:      int,
    eta:       float = ETA,
    dims:      tuple = DIMS,
    dir_name:  str   = DIR_NAME,
):
    """
    Load the ripser diagrams for one (arch, activation, seed) triple and
    return a 1-D array of shape (n_layers_total,).

    Index 0  = input layer point cloud.
    Index 1..n_hidden = hidden layers (in forward order).
    (No output-layer diagram is saved, matching the PCN convention where
    the output transition is included via include_output=True.)
    """
    rip_dir = RIPSER_ROOT / dir_name / arch_name / act_name / f'seed_{seed}'

    # --- input layer ---
    input_dgm = load_pickle(rip_dir / 'input_layer.pkl')
    betti_input = sum(_betti_number_from_diagrams(input_dgm, d, eta) for d in dims)

    # --- hidden layers (list stored in model.pkl) ---
    layer_dgms = load_pickle(rip_dir / 'model.pkl')   # list of diagram lists
    betti_hidden = [
        sum(_betti_number_from_diagrams(dgm, d, eta) for d in dims)
        for dgm in layer_dgms
    ]

    return np.array([betti_input] + betti_hidden, dtype=float)


def get_betti_mat_ann(
    arch_name: str,
    act_name:  str,
    n_seeds:   int   = N_SEEDS,
    eta:       float = ETA,
    dims:      tuple = DIMS,
    dir_name:  str   = DIR_NAME,
):
    """
    Returns betti_mat of shape (K, n_layers_total) where K = number of
    successfully loaded seeds.  Rows with missing ripser files are skipped
    with a warning.
    """
    rows = []
    for seed in range(n_seeds):
        rip_path = RIPSER_ROOT / dir_name / arch_name / act_name / f'seed_{seed}' / 'model.pkl'
        if not rip_path.exists():
            print(f'  [skip] missing ripser file: {rip_path}')
            continue
        try:
            curve = get_betti_curve_for_seed(arch_name, act_name, seed, eta, dims, dir_name)
            rows.append(curve)
        except Exception as e:
            print(f'  [error] seed {seed}: {e}')
    if not rows:
        raise RuntimeError(f'No betti curves loaded for {arch_name}/{act_name}')
    return np.vstack(rows)   # (K, n_layers_total)


def _com_of_drops_one_seed(
    betti_curve,
    use_running_min: bool = True,
    include_output:  bool = True,
    no_drop_value         = None,
):
    """
    Compute a single scalar Center-of-Mass (COM) for one model run.

    betti_curve : 1-D array of shape (n_layers_total,).
                  Index 0 = input layer; subsequent indices = hidden layers.
    use_running_min : if True, use monotone-non-increasing envelope
                      (robust to transient rebounds).
    include_output  : if True, the final hidden→output transition is
                      counted (mirrors include_output in the PCN version).
    no_drop_value   : sentinel returned when there are no simplifying drops.
                      Defaults to n_layers_total (a 'worst-case' value).

    Returns
    -------
    float  – weighted mean transition index where drops occur
             (lower = earlier simplification).
    """
    beta          = np.asarray(betti_curve, dtype=float)
    n_layers_total = beta.shape[0]
    max_transition = (n_layers_total - 1) if include_output else (n_layers_total - 2)

    if max_transition < 1:
        return float(no_drop_value if no_drop_value is not None else n_layers_total)

    beta_use = beta[: max_transition + 1]
    if use_running_min:
        beta_use = np.minimum.accumulate(beta_use)  # monotone non-increasing

    drops = beta_use[:-1] - beta_use[1:]
    if not use_running_min:
        drops = np.maximum(0.0, drops)

    total_drop = float(np.sum(drops))
    if total_drop <= 0.0:
        return float(no_drop_value if no_drop_value is not None else n_layers_total)

    ell = np.arange(1, max_transition + 1, dtype=float)
    return float(np.dot(ell, drops) / total_drop)


print('COM helpers defined.')


In [ ]:
# Cell 11: get_com  –  compute COM for every seed of one (arch, activation) pair

def get_com(
    arch_name:       str,
    act_name:        str,
    n_seeds:         int   = N_SEEDS,
    dir_name:        str   = DIR_NAME,
    eta:             float = ETA,
    dims:            tuple = DIMS,
    use_running_min: bool  = True,
    include_output:  bool  = True,
):
    """
    Returns a 1-D numpy array of length K (number of successfully loaded
    seeds) where each entry is the COM scalar for that seed.

    Smaller COM  => topological simplification happens earlier in the network.
    Larger  COM  => simplification is pushed toward the output end.
    """
    betti_mat = get_betti_mat_ann(
        arch_name  = arch_name,
        act_name   = act_name,
        n_seeds    = n_seeds,
        eta        = eta,
        dims       = dims,
        dir_name   = dir_name,
    )                                  # (K, n_layers_total)
    K, n_layers_total = betti_mat.shape
    no_drop_value = float(n_layers_total)   # sentinel: 'no simplification'

    com = np.empty(K, dtype=float)
    for i in range(K):
        com[i] = _com_of_drops_one_seed(
            betti_mat[i],
            use_running_min = use_running_min,
            include_output  = include_output,
            no_drop_value   = no_drop_value,
        )
    return com


print('get_com defined.')


In [ ]:
# Cell 12: visualize_com  –  violin plot of COM across architectures / activations

def visualize_com(
    arch_act_pairs:  list,            # list of (arch_name, act_name) tuples
    n_seeds:         int   = N_SEEDS,
    dir_name:        str   = DIR_NAME,
    eta:             float = ETA,
    dims:            tuple = DIMS,
    use_running_min: bool  = True,
    include_output:  bool  = True,
    title:           str   = '',
    x_labels:        list  = None,    # custom x-tick labels (len must match pairs)
    x_axis_label:    str   = 'Architecture / Activation',
    save:            bool  = False,
    filename:        str   = None,
    figsize:         tuple = (12, 6),
):
    """
    For each (arch_name, act_name) pair, loads Betti curves via
    get_betti_mat_ann(), computes per-seed COM scalars, then draws a
    violin plot.  Prints the per-group mean COM for quick inspection.

    Parameters
    ----------
    arch_act_pairs : e.g. [('30x8','relu'), ('30x8','tanh'), ('24x8','relu')]
    x_labels       : optional list of display strings (same order as pairs).
                     Defaults to 'arch/act' strings.

    Interpretation
    --------------
    Smaller COM => earlier topological simplification in the network.
    """
    from matplotlib.ticker import MaxNLocator

    pair_labels = [
        (x_labels[i] if x_labels is not None else f'{a}/{act}')
        for i, (a, act) in enumerate(arch_act_pairs)
    ]

    rows = []
    for label, (arch_name, act_name) in zip(pair_labels, arch_act_pairs):
        print(f'Computing COM: {arch_name} | {act_name} ...', end=' ', flush=True)
        try:
            com_vals = get_com(
                arch_name       = arch_name,
                act_name        = act_name,
                n_seeds         = n_seeds,
                dir_name        = dir_name,
                eta             = eta,
                dims            = dims,
                use_running_min = use_running_min,
                include_output  = include_output,
            )
            print(f'mean COM = {np.mean(com_vals):.4f}  (K={len(com_vals)} seeds)')
            for v in com_vals:
                rows.append({'group': label, 'COM': float(v)})
        except Exception as e:
            print(f'SKIPPED ({e})')

    if not rows:
        print('No data to plot.')
        return

    df = pd.DataFrame(rows)
    order = [lbl for lbl in pair_labels if lbl in df['group'].unique()]

    fig, ax = plt.subplots(figsize=figsize)
    sns.violinplot(
        data      = df,
        x         = 'group',
        y         = 'COM',
        order     = order,
        inner     = 'box',
        linewidth = 0.75,
        ax        = ax,
    )

    ax.set_xlabel(x_axis_label, fontsize=14, labelpad=10)
    ax.set_ylabel('COM', fontsize=14, labelpad=6)

    # Build dimension string for title  (e.g. β₀  or  β₀ + β₁)
    betti_str = r'$\beta_{' + str(dims[0]) + r'}$'
    for d in dims[1:]:
        betti_str += r' + $\beta_{' + str(d) + r'}$'
    ax.set_title(
        (title if title else f'COM of {betti_str} drops') + f',  η={eta}',
        fontsize=16,
    )

    rotate = 30 if len(order) > 6 else 0
    ax.tick_params(axis='x', rotation=rotate, labelsize=11)
    if rotate:
        for lbl in ax.get_xticklabels():
            lbl.set_ha('right')

    plt.tight_layout()

    if save:
        dim_str = ''.join(str(d) for d in dims)
        out_path = FIG_ROOT / f'COM/{filename}_B{dim_str}_eta{eta}.png'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        print(f'Saved → {out_path}')

    plt.show()


print('visualize_com defined.')


In [ ]:
# Cell 13: compute & plot COM  –  example calls
#
# Adjust arch_act_pairs, eta, dims etc. to match what you want to compare.
# The cells below give three common usage patterns.

# ── Pattern A: compare architectures for one activation function ──────────────
arch_act_pairs_A = [(arch, 'relu') for arch in ARCHS]
x_labels_A = list(ARCHS.keys())

visualize_com(
    arch_act_pairs  = arch_act_pairs_A,
    x_labels        = x_labels_A,
    x_axis_label    = 'Architecture  (relu)',
    title           = 'COM by architecture  –  relu',
    eta             = ETA,
    dims            = DIMS,
    use_running_min = USE_RUNNING_MIN,
    include_output  = INCLUDE_OUTPUT,
    save            = False,
    filename        = 'arch_relu',
)


In [ ]:
# ── Pattern B: compare activation functions for one architecture ─────────────
arch_act_pairs_B = [('30x8', act) for act in ACTIVATIONS]
x_labels_B = ACTIVATIONS

visualize_com(
    arch_act_pairs  = arch_act_pairs_B,
    x_labels        = x_labels_B,
    x_axis_label    = 'Activation function  (30x8)',
    title           = 'COM by activation function  –  30×8',
    eta             = ETA,
    dims            = DIMS,
    use_running_min = USE_RUNNING_MIN,
    include_output  = INCLUDE_OUTPUT,
    save            = False,
    filename        = '30x8_activations',
)


In [ ]:
# ── Pattern C: full grid  (all arches × all activations) ─────────────────────
arch_act_pairs_C = [(arch, act) for arch in ARCHS for act in ACTIVATIONS]
x_labels_C = [f'{arch}\n{act}' for arch, act in arch_act_pairs_C]

visualize_com(
    arch_act_pairs  = arch_act_pairs_C,
    x_labels        = x_labels_C,
    x_axis_label    = 'Architecture / Activation',
    title           = 'COM – full grid',
    eta             = ETA,
    dims            = DIMS,
    use_running_min = USE_RUNNING_MIN,
    include_output  = INCLUDE_OUTPUT,
    figsize         = (18, 6),
    save            = False,
    filename        = 'full_grid',
)


In [ ]:
# Debug Cell A: inspect raw ripser file structure for one (arch, act, seed)
# -----------------------------------------------------------------------
# What we check:
#   1. Does the ripser directory exist?
#   2. What files are inside it?
#   3. What does model.pkl actually contain (type, length, per-layer types)?
#   4. What does input_layer.pkl contain?

_DBG_ARCH = '30x8'
_DBG_ACT  = 'relu'
_DBG_SEED = 0

rip_dir = RIPSER_ROOT / DIR_NAME / _DBG_ARCH / _DBG_ACT / f'seed_{_DBG_SEED}'
print('=== Ripser directory ===')
print('path   :', rip_dir)
print('exists :', rip_dir.exists())

if rip_dir.exists():
    files = sorted(rip_dir.iterdir())
    print('files  :', [p.name for p in files])
    print()

    # --- input_layer.pkl ---
    inp_path = rip_dir / 'input_layer.pkl'
    if inp_path.exists():
        inp_dgm = load_pickle(inp_path)
        print('=== input_layer.pkl ===')
        print('type         :', type(inp_dgm))
        if isinstance(inp_dgm, list):
            print('len          :', len(inp_dgm))   # should be maxdim+1
            for di, d in enumerate(inp_dgm):
                import numpy as _np
                d = _np.asarray(d)
                print(f'  dim {di}: shape={d.shape}, dtype={d.dtype}')
                if len(d):
                    lifetimes = d[:, 1] - d[:, 0]
                    finite = _np.isfinite(lifetimes)
                    print(f'         finite pairs={finite.sum()}, '
                          f'max_lifetime={lifetimes[finite].max() if finite.any() else "n/a":.4f}')
        else:
            print('  (unexpected type)')
    print()

    # --- model.pkl ---
    mdl_path = rip_dir / 'model.pkl'
    if mdl_path.exists():
        model_dgms = load_pickle(mdl_path)
        print('=== model.pkl ===')
        print('outer type   :', type(model_dgms))
        print('outer len    :', len(model_dgms), ' <-- should equal n_hidden_layers')
        for li, layer_entry in enumerate(model_dgms):
            print(f'  layer {li}: type={type(layer_entry)}', end='')
            if isinstance(layer_entry, list):
                print(f', len={len(layer_entry)} (one entry per homology dim)')
                for di, dgm in enumerate(layer_entry):
                    import numpy as _np
                    dgm = _np.asarray(dgm)
                    print(f'    dim {di}: shape={dgm.shape}', end='')
                    if dgm.size:
                        lt = dgm[:, 1] - dgm[:, 0]
                        finite = _np.isfinite(lt)
                        print(f', finite_pairs={finite.sum()}, '
                              f'max_lt={lt[finite].max() if finite.any() else "n/a":.4f}', end='')
                    print()
            else:
                print()
else:
    print('Directory not found. Check RIPSER_ROOT / DIR_NAME path.')
    print('RIPSER_ROOT:', RIPSER_ROOT)
    print('DIR_NAME   :', DIR_NAME)


In [ ]:
# Debug Cell B: trace the full Betti-curve computation for one seed
# -----------------------------------------------------------------------
# Shows:
#   - Raw Betti numbers at each layer (before running-min)
#   - Running-min envelope
#   - Per-transition drops
#   - COM scalar
#
# If all Betti numbers are 0 → eta is too large (no pairs survive the threshold).
# If Betti numbers are flat (no drops) → COM = sentinel.

import numpy as _np

_DBG_ARCH = '30x8'
_DBG_ACT  = 'relu'
_DBG_SEED = 0

# ---- 1. Raw Betti numbers at every eta (scan) --------------------------------
print('=== Betti numbers vs eta (dim 0, input layer) ===')
rip_dir = RIPSER_ROOT / DIR_NAME / _DBG_ARCH / _DBG_ACT / f'seed_{_DBG_SEED}'
inp_dgm = load_pickle(rip_dir / 'input_layer.pkl')
model_dgms = load_pickle(rip_dir / 'model.pkl')

# All finite lifetimes in dim-0 across all layers
all_lifetimes = []
for dgm_list in [inp_dgm] + model_dgms:
    if 0 < len(dgm_list):
        d = _np.asarray(dgm_list[0])
        lt = d[:, 1] - d[:, 0]
        all_lifetimes.extend(lt[_np.isfinite(lt)].tolist())

all_lifetimes = _np.array(sorted(all_lifetimes, reverse=True))
print(f'Total finite dim-0 lifetimes across all layers: {len(all_lifetimes)}')
if len(all_lifetimes):
    print(f'Top-10 lifetimes: {_np.round(all_lifetimes[:10], 4)}')
    print(f'Current eta = {ETA}  -->  pairs surviving = {(all_lifetimes > ETA).sum()}')
    # Suggest a better eta
    median_lt = float(_np.median(all_lifetimes))
    p25_lt    = float(_np.percentile(all_lifetimes, 25))
    print(f'Suggested eta values:  median={median_lt:.4f}  p25={p25_lt:.4f}')
print()

# ---- 2. Betti curve at current ETA ------------------------------------------
print(f'=== Betti curve at eta={ETA}, dims={DIMS} ===')
curve = get_betti_curve_for_seed(_DBG_ARCH, _DBG_ACT, _DBG_SEED, eta=ETA, dims=DIMS)
print('Raw Betti curve (index=layer, 0=input):', curve)

running_min = _np.minimum.accumulate(curve)
print('Running min:                           ', running_min)

drops = running_min[:-1] - running_min[1:]
print('Drops (Δ per transition):              ', drops)
print('Total drop D:                          ', drops.sum())
if drops.sum() > 0:
    ell = _np.arange(1, len(drops) + 1, dtype=float)
    print('COM:                                   ', float(_np.dot(ell, drops) / drops.sum()))
else:
    print('COM: sentinel (no drops) =', len(curve))
print()

# ---- 3. Try scanning eta to find a range where Betti drops exist -------------
print('=== Betti curve scan across eta values ===')
print(f'{"eta":>8}  {"B[0]":>6}  {"B[1]":>6}  ...  {"total_drop":>12}  {"COM":>8}')
eta_candidates = _np.unique(_np.concatenate([
    [0.01, 0.05, 0.1, 0.2, 0.5],
    _np.percentile(all_lifetimes, [10, 25, 50, 75]) if len(all_lifetimes) else [],
]))
for et in eta_candidates:
    c = get_betti_curve_for_seed(_DBG_ARCH, _DBG_ACT, _DBG_SEED, eta=float(et), dims=DIMS)
    rm = _np.minimum.accumulate(c)
    dr = rm[:-1] - rm[1:]
    td = dr.sum()
    com_val = (float(_np.dot(_np.arange(1, len(dr)+1, dtype=float), dr) / td)
               if td > 0 else len(c))
    print(f'{et:>8.4f}  {"  ".join(f"{int(v):>4}" for v in c[:3])}  ...  '
          f'{td:>12.1f}  {com_val:>8.4f}')


In [ ]:
# Debug Cell C: verify activation arrays look sensible
# -----------------------------------------------------------------------
# Checks that activations were saved and have reasonable statistics.
# Degenerate activations (all-zero, collapsed) cause trivial topology.

import numpy as _np

_DBG_ARCH = '30x8'
_DBG_ACT  = 'relu'
_DBG_SEED = 0

act_dir = ACT_ROOT / _DBG_ARCH / _DBG_ACT / f'seed_{_DBG_SEED}'
print('Activation dir:', act_dir)
print('Exists        :', act_dir.exists())

if act_dir.exists():
    pkl_files = sorted(act_dir.glob('*.pkl'))
    print(f'Found {len(pkl_files)} pkl files:', [p.name for p in pkl_files])
    print()
    for p in pkl_files:
        A = load_pickle(p)
        A = _np.asarray(A, dtype=float)
        frac_zero  = float((_np.abs(A) < 1e-8).mean())
        frac_finite = float(_np.isfinite(A).mean())
        print(f'{p.name:30s}  shape={str(A.shape):15s}  '
              f'mean={A.mean():.4f}  std={A.std():.4f}  '
              f'frac_zero={frac_zero:.2%}  frac_finite={frac_finite:.2%}')
    print()
    print('NOTE: High frac_zero (>80%) with ReLU is normal but extreme collapse')
    print('      (>99%) may produce trivial topology. All-zero layers are a red flag.')


In [ ]:
# Debug Cell D: corrected _com_of_drops_one_seed matching the paper definition
# -----------------------------------------------------------------------
# The paper defines COM = sum_{ell=1}^{L+1} ell * p(ell)
# where ell indexes *transitions* (1-indexed), NOT layers.
#
# For an 8-hidden-layer network:
#   betti_curve has 9 entries  (index 0=input, 1..8=hidden layers)
#   transitions 1..8 connect consecutive layers
#   --> COM range is [1, 8]  (never 9, never 0)
#
# The sentinel in the previous version was n_layers_total=9, which is
# OUTSIDE the valid COM range and causes the flat line at y=9.
#
# Fix: use np.nan as the sentinel so invalid seeds are visible/droppable,
# and verify the indexing is 1-based on transitions.

import numpy as _np

def _com_of_drops_one_seed_v2(
    betti_curve,
    use_running_min: bool = True,
    include_output:  bool = True,
    no_drop_value         = float('nan'),   # nan = 'no simplification detected'
):
    """
    Corrected COM matching the paper definition:

        COM = sum_{ell=1}^{L+1}  ell * Delta(ell) / D

    where ell indexes *transitions* (1-indexed).

    betti_curve : 1-D array of length n_layers_total.
                  Index 0 = input layer,  1..L = hidden layers.
    include_output = True  → last transition counted (ell goes to L, i.e. max_transition = L).
    include_output = False → skip last hidden→output transition.

    Valid COM range: [1, max_transition]  (never equals n_layers_total as a sentinel).
    Returns no_drop_value (default nan) when D=0.
    """
    beta           = _np.asarray(betti_curve, dtype=float)
    n_layers_total = beta.shape[0]              # e.g. 9 for 8 hidden layers
    # max_transition: number of transitions we consider
    # transition ell connects layer (ell-1) -> layer ell
    # With include_output=True and n_layers_total=9:
    #   ell in {1,2,...,8}  → beta indices 0..8  → max_transition=8
    max_transition = (n_layers_total - 1) if include_output else (n_layers_total - 2)
    if max_transition < 1:
        return float(no_drop_value)

    beta_use = beta[: max_transition + 1]       # shape (max_transition+1,)
    if use_running_min:
        beta_use = _np.minimum.accumulate(beta_use)

    # Delta(ell) for ell = 1 .. max_transition
    drops = beta_use[:-1] - beta_use[1:]        # shape (max_transition,)
    if not use_running_min:
        drops = _np.maximum(0.0, drops)

    D = float(_np.sum(drops))
    if D <= 0.0:
        return float(no_drop_value)

    # ell is 1-indexed (transition index)
    ell = _np.arange(1, max_transition + 1, dtype=float)
    return float(_np.dot(ell, drops) / D)


def get_com_v2(
    arch_name:       str,
    act_name:        str,
    n_seeds:         int   = N_SEEDS,
    dir_name:        str   = DIR_NAME,
    eta:             float = ETA,
    dims:            tuple = DIMS,
    use_running_min: bool  = True,
    include_output:  bool  = True,
    drop_nan:        bool  = True,    # drop seeds with no simplification
):
    """
    Like get_com but uses _com_of_drops_one_seed_v2.
    Returns array of COM values in [1, L] (nan-free if drop_nan=True).
    """
    betti_mat = get_betti_mat_ann(
        arch_name=arch_name, act_name=act_name, n_seeds=n_seeds,
        eta=eta, dims=dims, dir_name=dir_name,
    )
    K = betti_mat.shape[0]
    com = _np.array([
        _com_of_drops_one_seed_v2(
            betti_mat[i],
            use_running_min=use_running_min,
            include_output=include_output,
        ) for i in range(K)
    ])
    if drop_nan:
        n_nan = int(_np.isnan(com).sum())
        if n_nan:
            print(f'  [{arch_name}/{act_name}] {n_nan}/{K} seeds had no drops (nan) → dropped')
        com = com[~_np.isnan(com)]
    return com


# ---- Quick sanity check: print COM for a few groups with both eta values ----
print('=== Sanity check: mean COM for 30x8/relu ===\n')
for et in [ETA, 0.1, 0.5]:
    c = get_com_v2('30x8', 'relu', eta=et)
    print(f'  eta={et:5.2f} -> n_valid={len(c)}, '
          f'mean_COM={_np.mean(c):.3f} (should be in [1, {N_SEEDS}])')
    if len(c):
        print(f'           COM values: {_np.round(c[:5], 2)} ...')


In [ ]:
# Debug Cell E: replot the full grid using get_com_v2 + corrected eta
# -----------------------------------------------------------------------
# After running Cells D, read the 'Suggested eta' from Cell B output and
# set FIXED_ETA below before running this cell.

import numpy as _np

FIXED_ETA = 0.1   # <-- update from Cell B 'Suggested eta' output

def visualize_com_v2(
    arch_act_pairs,
    eta             = FIXED_ETA,
    dims            = DIMS,
    n_seeds         = N_SEEDS,
    dir_name        = DIR_NAME,
    use_running_min = True,
    include_output  = True,
    x_labels        = None,
    x_axis_label    = 'Architecture / Activation',
    title           = '',
    figsize         = (18, 6),
    save            = False,
    filename        = None,
):
    rows = []
    pair_labels = [
        (x_labels[i] if x_labels is not None else f'{a}/{act}')
        for i, (a, act) in enumerate(arch_act_pairs)
    ]
    for label, (arch, act) in zip(pair_labels, arch_act_pairs):
        com_vals = get_com_v2(arch, act, eta=eta, dims=dims,
                              n_seeds=n_seeds, dir_name=dir_name,
                              use_running_min=use_running_min,
                              include_output=include_output)
        print(f'{arch}/{act}: n_valid={len(com_vals)}, '
              f'mean={_np.mean(com_vals):.3f} '
              f'(range [{com_vals.min():.2f}, {com_vals.max():.2f}])')
        for v in com_vals:
            rows.append({'group': label, 'COM': float(v)})

    if not rows:
        print('No data. Increase eta or check ripser files.')
        return

    df = pd.DataFrame(rows)
    order = [l for l in pair_labels if l in df['group'].unique()]

    # Total hidden layers (infer from first arch)
    first_arch = arch_act_pairs[0][0]
    L = len(ARCHS[first_arch])  # number of hidden layers

    fig, ax = plt.subplots(figsize=figsize)
    sns.violinplot(data=df, x='group', y='COM', order=order,
                   inner='box', linewidth=0.75, ax=ax)

    # Draw valid COM range as a reference band
    ax.axhline(1, color='gray', lw=0.8, ls='--', alpha=0.5, label='COM=1 (earliest)')
    ax.axhline(L, color='gray', lw=0.8, ls=':', alpha=0.5, label=f'COM={L} (latest)')
    ax.set_ylim(0, L + 1)
    ax.legend(fontsize=9)

    betti_str = r'$\beta_{' + str(dims[0]) + r'}$'
    ax.set_title((title if title else f'COM of {betti_str} drops') + f',  η={eta}', fontsize=16)
    ax.set_xlabel(x_axis_label, fontsize=14, labelpad=10)
    ax.set_ylabel('COM  (transition index, 1-based)', fontsize=13, labelpad=6)

    rotate = 30 if len(order) > 6 else 0
    ax.tick_params(axis='x', rotation=rotate, labelsize=10)
    if rotate:
        for lbl in ax.get_xticklabels(): lbl.set_ha('right')

    plt.tight_layout()
    if save and filename:
        dim_str = ''.join(str(d) for d in dims)
        out_path = FIG_ROOT / f'COM/{filename}_B{dim_str}_eta{eta}.png'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        print(f'Saved → {out_path}')
    plt.show()


# Run on full grid
arch_act_pairs_C = [(arch, act) for arch in ARCHS for act in ACTIVATIONS]
x_labels_C = [f'{arch}\n{act}' for arch, act in arch_act_pairs_C]

visualize_com_v2(
    arch_act_pairs = arch_act_pairs_C,
    x_labels       = x_labels_C,
    eta            = FIXED_ETA,
    title          = f'COM – full grid (v2, η={FIXED_ETA})',
    save           = False,
    filename       = 'full_grid_v2',
)


In [ ]:
# Cell F: verify the root cause and show the corrected Betti curve
# -----------------------------------------------------------------------
# The problem: input_layer.pkl is a 2-D point cloud → β₀(input)=0 at eta=2.5
# because all 14 dim-0 lifetimes are ≤ 1.55 < 2.5.
# The running-min locks to 0 immediately and never drops again.
#
# The fix: build the Betti curve from HIDDEN LAYERS ONLY (model.pkl entries),
# i.e. ℓ=0..L-1.  Transitions are then ℓ=1..L (between hidden layers),
# matching the paper's sum.

import numpy as _np

_DBG_ARCH = '30x8'
_DBG_ACT  = 'relu'
_DBG_SEED = 0

rip_dir     = RIPSER_ROOT / DIR_NAME / _DBG_ARCH / _DBG_ACT / f'seed_{_DBG_SEED}'
inp_dgm     = load_pickle(rip_dir / 'input_layer.pkl')
model_dgms  = load_pickle(rip_dir / 'model.pkl')   # list of 8 diagram-lists

print('=== Old curve (input layer included) ===')
betti_input = sum(_betti_number_from_diagrams(inp_dgm, d, ETA) for d in DIMS)
betti_hidden = [sum(_betti_number_from_diagrams(dgm, d, ETA) for d in DIMS)
                for dgm in model_dgms]
old_curve = _np.array([betti_input] + betti_hidden, dtype=float)
old_rm    = _np.minimum.accumulate(old_curve)
print('Curve      :', old_curve)
print('Running min:', old_rm)
print('Drops      :', old_rm[:-1] - old_rm[1:])
print(f'→ β₀(input)={betti_input}  locks running-min to 0 → no drops ever')
print()

print('=== New curve (hidden layers only, model.pkl) ===')
new_curve = _np.array(betti_hidden, dtype=float)   # length = L = 8
new_rm    = _np.minimum.accumulate(new_curve)
new_drops = new_rm[:-1] - new_rm[1:]               # L-1 = 7 transitions
print('Curve      :', new_curve)
print('Running min:', new_rm)
print('Drops Δ(ℓ) :', new_drops)
D = new_drops.sum()
print('Total D    :', D)
if D > 0:
    ell = _np.arange(1, len(new_drops) + 1, dtype=float)
    com = float(_np.dot(ell, new_drops) / D)
    print(f'COM        : {com:.4f}  (valid range [1, {len(new_drops)}])')
else:
    print('COM: still no drops — try a lower eta')
print()

# Scan eta on the corrected (hidden-only) curve
print('=== eta scan on hidden-only curve ===')
print(f'{"eta":>8}  {"curve[:4]":^28}  {"D":>8}  {"COM":>8}')
for et in [0.05, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
    c = _np.array([
        sum(_betti_number_from_diagrams(dgm, d, et) for d in DIMS)
        for dgm in model_dgms
    ], dtype=float)
    rm = _np.minimum.accumulate(c)
    dr = rm[:-1] - rm[1:]
    D  = dr.sum()
    if D > 0:
        ell = _np.arange(1, len(dr) + 1, dtype=float)
        com_val = float(_np.dot(ell, dr) / D)
    else:
        com_val = float('nan')
    c_str = '  '.join(f'{int(v):3d}' for v in c[:4])
    print(f'{et:>8.2f}  [{c_str} ...]  {D:>8.1f}  {com_val:>8.4f}')


In [ ]:
# Cell G: corrected get_betti_curve_v3 / get_com_v3 / visualize_com_v3
# -----------------------------------------------------------------------
# Key change: Betti curve is built from HIDDEN LAYERS ONLY (model.pkl).
# The input point cloud is excluded from the running-min because its
# β₀ is limited by n_perm (=15) and collapses to 0 at any eta larger
# than the max pairwise distance in a 2-D probe set, skewing the min.
#
# With L hidden layers the curve has length L, transitions ℓ=1..L-1
# (between consecutive hidden layers), and COM ∈ [1, L-1].
# Setting include_output=True adds the final hidden→output transition,
# giving COM ∈ [1, L].

import numpy as _np

def get_betti_curve_v3(
    arch_name: str,
    act_name:  str,
    seed:      int,
    eta:       float = ETA,
    dims:      tuple = DIMS,
    dir_name:  str   = DIR_NAME,
):
    """
    Returns a 1-D array of length L (number of hidden layers).
    Entry ℓ = β(hidden layer ℓ).  The input point cloud is excluded.
    """
    rip_dir   = RIPSER_ROOT / dir_name / arch_name / act_name / f'seed_{seed}'
    layer_dgms = load_pickle(rip_dir / 'model.pkl')   # list of L diagram-lists
    return _np.array([
        sum(_betti_number_from_diagrams(dgm, d, eta) for d in dims)
        for dgm in layer_dgms
    ], dtype=float)


def get_betti_mat_v3(
    arch_name: str,
    act_name:  str,
    n_seeds:   int   = N_SEEDS,
    eta:       float = ETA,
    dims:      tuple = DIMS,
    dir_name:  str   = DIR_NAME,
):
    """Returns (K, L) matrix — K seeds, L hidden layers."""
    rows = []
    for seed in range(n_seeds):
        rip_path = RIPSER_ROOT / dir_name / arch_name / act_name / f'seed_{seed}' / 'model.pkl'
        if not rip_path.exists():
            continue
        try:
            rows.append(get_betti_curve_v3(arch_name, act_name, seed, eta, dims, dir_name))
        except Exception as e:
            print(f'  [error] seed {seed}: {e}')
    if not rows:
        raise RuntimeError(f'No betti curves loaded for {arch_name}/{act_name}')
    return _np.vstack(rows)   # (K, L)


def _com_v3(betti_curve, use_running_min=True, include_output=True):
    """
    COM matching the paper definition exactly.

    betti_curve : length-L array (hidden layers only).
    Transitions: ℓ = 1 .. L-1  (between consecutive hidden layers).
    include_output=True adds transition L (last hidden → output),
      giving COM ∈ [1, L].  False gives COM ∈ [1, L-1].
    Returns nan when D=0 (no simplification detected).
    """
    beta = _np.asarray(betti_curve, dtype=float)
    L    = beta.shape[0]                           # number of hidden layers
    # max_transition: how many consecutive-layer transitions to consider
    # ℓ=1 means layers[0]→layers[1], …, ℓ=L-1 means layers[L-2]→layers[L-1]
    # ℓ=L means layers[L-1]→output  (only if include_output=True)
    max_t = L if include_output else (L - 1)
    if max_t < 1:
        return float('nan')

    b = beta[:max_t + 1] if include_output else beta[:max_t + 1]
    # When include_output=True we have L values (all hidden layers),
    # transitions are between consecutive pairs → L-1 drops internally,
    # plus we append a notional β=0 at the output to capture the final drop.
    if include_output:
        b = _np.append(beta, 0.0)   # output layer β = 0 by convention
        b = b                        # length L+1: indices 0..L
    else:
        b = beta                     # length L

    if use_running_min:
        b = _np.minimum.accumulate(b)

    drops = b[:-1] - b[1:]          # Δ(ℓ) for ℓ=1..max_t
    drops = _np.maximum(0.0, drops)
    D     = float(drops.sum())
    if D <= 0.0:
        return float('nan')

    ell = _np.arange(1, len(drops) + 1, dtype=float)  # 1-indexed transitions
    return float(_np.dot(ell, drops) / D)


def get_com_v3(
    arch_name:       str,
    act_name:        str,
    n_seeds:         int   = N_SEEDS,
    dir_name:        str   = DIR_NAME,
    eta:             float = ETA,
    dims:            tuple = DIMS,
    use_running_min: bool  = True,
    include_output:  bool  = True,
    drop_nan:        bool  = True,
):
    betti_mat = get_betti_mat_v3(arch_name, act_name, n_seeds, eta, dims, dir_name)
    K = betti_mat.shape[0]
    com = _np.array([
        _com_v3(betti_mat[i], use_running_min=use_running_min,
                include_output=include_output)
        for i in range(K)
    ])
    if drop_nan:
        n_nan = int(_np.isnan(com).sum())
        if n_nan:
            print(f'  [{arch_name}/{act_name}] {n_nan}/{K} seeds had no drops → dropped')
        com = com[~_np.isnan(com)]
    return com


# Quick sanity check
print('=== Sanity check: 30x8/relu ===')
for et in [ETA, 1.0, 0.5, 0.1]:
    c = get_com_v3('30x8', 'relu', eta=et)
    if len(c):
        print(f'  eta={et:.2f}  n_valid={len(c):3d}  '
              f'mean={_np.mean(c):.3f}  '
              f'range=[{c.min():.2f}, {c.max():.2f}]')
    else:
        print(f'  eta={et:.2f}  → still no drops (lower eta needed)')


In [ ]:
# Cell H: final corrected full-grid COM violin plot
# -----------------------------------------------------------------------
# Uses get_com_v3 (hidden-layers-only Betti curve, nan sentinel).
# Set PLOT_ETA to the value from Cell F's eta scan where drops are non-zero.

import numpy as _np

PLOT_ETA = ETA   # adjust if Cell F's scan shows a better value

def visualize_com_v3(
    arch_act_pairs,
    eta             = PLOT_ETA,
    dims            = DIMS,
    n_seeds         = N_SEEDS,
    dir_name        = DIR_NAME,
    use_running_min = USE_RUNNING_MIN,
    include_output  = INCLUDE_OUTPUT,
    x_labels        = None,
    x_axis_label    = 'Architecture / Activation',
    title           = '',
    figsize         = (18, 6),
    save            = False,
    filename        = None,
):
    rows   = []
    pair_labels = [
        (x_labels[i] if x_labels else f'{a}/{act}')
        for i, (a, act) in enumerate(arch_act_pairs)
    ]

    for label, (arch, act) in zip(pair_labels, arch_act_pairs):
        try:
            com_vals = get_com_v3(
                arch_name=arch, act_name=act, eta=eta, dims=dims,
                n_seeds=n_seeds, dir_name=dir_name,
                use_running_min=use_running_min,
                include_output=include_output,
            )
            if len(com_vals) == 0:
                print(f'  {arch}/{act}: no valid seeds at eta={eta}')
                continue
            print(f'  {arch}/{act}: n={len(com_vals):3d}  '
                  f'mean={_np.mean(com_vals):.3f}  '
                  f'[{com_vals.min():.2f}, {com_vals.max():.2f}]')
            for v in com_vals:
                rows.append({'group': label, 'COM': float(v)})
        except Exception as e:
            print(f'  {arch}/{act}: SKIPPED ({e})')

    if not rows:
        print('No data. Lower eta or check ripser files.')
        return

    df    = pd.DataFrame(rows)
    order = [l for l in pair_labels if l in df['group'].unique()]

    # Infer L from first arch
    L = len(ARCHS[arch_act_pairs[0][0]])
    max_com = L if include_output else L - 1

    fig, ax = plt.subplots(figsize=figsize)
    sns.violinplot(
        data=df, x='group', y='COM', order=order,
        inner='box', linewidth=0.75, ax=ax,
    )

    # Reference lines for valid COM range
    ax.axhline(1,       color='steelblue', lw=1.0, ls='--', alpha=0.6,
               label='COM = 1  (earliest)')
    ax.axhline(max_com, color='tomato',    lw=1.0, ls='--', alpha=0.6,
               label=f'COM = {max_com}  (latest)')
    ax.set_ylim(0, max_com + 1)
    ax.legend(fontsize=9)

    betti_str = r'$\beta_{' + str(dims[0]) + r'}$'
    for d in dims[1:]:
        betti_str += r' + $\beta_{' + str(d) + r'}$'
    ax.set_title(
        (title if title else f'COM of {betti_str} drops') + f',  η={eta}',
        fontsize=16,
    )
    ax.set_xlabel(x_axis_label, fontsize=14, labelpad=10)
    ax.set_ylabel('COM  (transition index, 1-based)', fontsize=13, labelpad=6)

    rotate = 30 if len(order) > 6 else 0
    ax.tick_params(axis='x', rotation=rotate, labelsize=10)
    if rotate:
        for lbl in ax.get_xticklabels():
            lbl.set_ha('right')

    plt.tight_layout()
    if save and filename:
        dim_str  = ''.join(str(d) for d in dims)
        out_path = FIG_ROOT / f'COM/{filename}_B{dim_str}_eta{eta}.png'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        print(f'Saved → {out_path}')
    plt.show()


# ── Run ──────────────────────────────────────────────────────────────────────
arch_act_pairs_C = [(arch, act) for arch in ARCHS for act in ACTIVATIONS]
x_labels_C = [f'{arch}\n{act}' for arch, act in arch_act_pairs_C]

visualize_com_v3(
    arch_act_pairs = arch_act_pairs_C,
    x_labels       = x_labels_C,
    eta            = PLOT_ETA,
    title          = f'COM – full grid',
    save           = True,
    filename       = 'full_grid_v3',
)



In [ ]:
# Cell I: COM by architecture
# For each activation function, plot a violin per architecture.
# This lets you ask: 'does a wider/narrower/tapered network simplify earlier?'

import numpy as _np

fig, axes = plt.subplots(
    1, len(ACTIVATIONS),
    figsize=(6 * len(ACTIVATIONS), 6),
    sharey=True,
)

# Infer valid COM range from first arch
_L       = len(next(iter(ARCHS.values())))
_max_com = _L if INCLUDE_OUTPUT else _L - 1

for ax, act_name in zip(axes, ACTIVATIONS):
    rows = []
    for arch_name in ARCHS:
        try:
            com_vals = get_com_v3(
                arch_name       = arch_name,
                act_name        = act_name,
                eta             = PLOT_ETA,
                dims            = DIMS,
                n_seeds         = N_SEEDS,
                dir_name        = DIR_NAME,
                use_running_min = USE_RUNNING_MIN,
                include_output  = INCLUDE_OUTPUT,
            )
            if len(com_vals) == 0:
                print(f'  {arch_name}/{act_name}: no valid seeds')
                continue
            print(f'  {arch_name}/{act_name}: n={len(com_vals):3d}  mean={_np.mean(com_vals):.3f}')
            for v in com_vals:
                rows.append({'arch': arch_name, 'COM': float(v)})
        except Exception as e:
            print(f'  {arch_name}/{act_name}: SKIPPED ({e})')

    if not rows:
        ax.set_title(f'{act_name}\n(no data)')
        continue

    df    = pd.DataFrame(rows)
    order = [a for a in ARCHS if a in df['arch'].unique()]

    sns.violinplot(
        data=df, x='arch', y='COM', order=order,
        inner='box', linewidth=0.75, ax=ax,
    )

    ax.axhline(1,        color='steelblue', lw=1.0, ls='--', alpha=0.5)
    ax.axhline(_max_com, color='tomato',    lw=1.0, ls='--', alpha=0.5)
    ax.set_ylim(0, _max_com + 1)
    ax.set_title(act_name, fontsize=14)
    ax.set_xlabel('Architecture', fontsize=12, labelpad=8)
    ax.set_ylabel('COM' if ax is axes[0] else '', fontsize=12)
    ax.tick_params(axis='x', rotation=30, labelsize=9)
    for lbl in ax.get_xticklabels():
        lbl.set_ha('right')

betti_str = r'$\beta_{' + str(DIMS[0]) + r'}$'
fig.suptitle(
    f'COM of {betti_str} drops by architecture,  η={PLOT_ETA}',
    fontsize=16, y=1.02,
)
plt.tight_layout()

# Optional save
_save = True
if _save:
    dim_str  = ''.join(str(d) for d in DIMS)
    out_path = FIG_ROOT / f'COM/by_arch_B{dim_str}_eta{PLOT_ETA}.png'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    print(f'Saved → {out_path}')

plt.show()


In [ ]:
# Cell J: COM by activation function
# For each architecture, plot a violin per activation function.
# This lets you ask: 'does relu vs tanh vs leaky_relu simplify at a different depth?'

import numpy as _np

n_archs = len(ARCHS)
ncols   = min(3, n_archs)
nrows   = math.ceil(n_archs / ncols)

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(5 * ncols, 5 * nrows),
    sharey=True,
    squeeze=False,
)
axes_flat = axes.flatten()

_L       = len(next(iter(ARCHS.values())))
_max_com = _L if INCLUDE_OUTPUT else _L - 1

for ax, arch_name in zip(axes_flat, ARCHS):
    rows = []
    for act_name in ACTIVATIONS:
        try:
            com_vals = get_com_v3(
                arch_name       = arch_name,
                act_name        = act_name,
                eta             = PLOT_ETA,
                dims            = DIMS,
                n_seeds         = N_SEEDS,
                dir_name        = DIR_NAME,
                use_running_min = USE_RUNNING_MIN,
                include_output  = INCLUDE_OUTPUT,
            )
            if len(com_vals) == 0:
                print(f'  {arch_name}/{act_name}: no valid seeds')
                continue
            print(f'  {arch_name}/{act_name}: n={len(com_vals):3d}  mean={_np.mean(com_vals):.3f}')
            for v in com_vals:
                rows.append({'activation': act_name, 'COM': float(v)})
        except Exception as e:
            print(f'  {arch_name}/{act_name}: SKIPPED ({e})')

    if not rows:
        ax.set_title(f'{arch_name}\n(no data)')
        continue

    df    = pd.DataFrame(rows)
    order = [a for a in ACTIVATIONS if a in df['activation'].unique()]

    sns.violinplot(
        data=df, x='activation', y='COM', order=order,
        inner='box', linewidth=0.75, ax=ax,
    )

    ax.axhline(1,        color='steelblue', lw=1.0, ls='--', alpha=0.5)
    ax.axhline(_max_com, color='tomato',    lw=1.0, ls='--', alpha=0.5)
    ax.set_ylim(0, _max_com + 1)
    ax.set_title(arch_name, fontsize=13)
    ax.set_xlabel('Activation', fontsize=11, labelpad=8)
    ax.set_ylabel('COM' if ax is axes_flat[0] else '', fontsize=11)
    ax.tick_params(axis='x', rotation=20, labelsize=10)

# Hide unused subplot panels
for ax in axes_flat[n_archs:]:
    ax.set_visible(False)

betti_str = r'$\beta_{' + str(DIMS[0]) + r'}$'
fig.suptitle(
    f'COM of {betti_str} drops by activation function,  η={PLOT_ETA}',
    fontsize=16, y=1.02,
)
plt.tight_layout()

# Optional save
_save = True
if _save:
    dim_str  = ''.join(str(d) for d in DIMS)
    out_path = FIG_ROOT / f'COM/by_act_B{dim_str}_eta{PLOT_ETA}.png'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    print(f'Saved → {out_path}')

plt.show()


In [ ]:
# Cell K: Betti curve per architecture — mean ± std over 30 seeds
# One figure per (arch, activation) pair.  Individual seed traces shown faint.
# Matches the PCN graph_betti_numbers style from Trainer.py.

import numpy as _np

def plot_betti_curve_ann(
    arch_name:          str,
    act_name:           str,
    eta:                float = PLOT_ETA,
    dims:               tuple = DIMS,
    n_seeds:            int   = N_SEEDS,
    dir_name:           str   = DIR_NAME,
    use_running_min:    bool  = False,      # False = raw curves (like PCN default)
    color:              str   = 'blue',
    plot_individual:    bool  = True,
    alpha_individual:   float = 0.12,
    lw_individual:      float = 1.0,
    lw_mean:            float = 2.0,
    marker:             str   = 's',
    figsize:            tuple = (10, 6),
    title:              str   = None,
    save:               bool  = False,
    filename:           str   = None,
):
    """
    Plot the Betti curve (hidden layers only) for one (arch, activation) pair.
    Shows all individual seed traces faintly, overlaid with the mean line
    and a ±1 std band — matching the PCN graph_betti_numbers style.

    X-axis: layer index  (0 = first hidden layer, L-1 = last hidden layer).
    Y-axis: β_{dim} (or sum of dims) at the given eta threshold.
    """
    # ---- build betti matrix (K, L) -----------------------------------------
    betti_mat = get_betti_mat_v3(
        arch_name       = arch_name,
        act_name        = act_name,
        n_seeds         = n_seeds,
        eta             = eta,
        dims            = dims,
        dir_name        = dir_name,
    )                                          # (K, L)

    if use_running_min:
        betti_mat = _np.minimum.accumulate(betti_mat, axis=1)

    K, L = betti_mat.shape
    mean_curve = betti_mat.mean(axis=0)        # (L,)
    std_curve  = betti_mat.std(axis=0, ddof=1) # (L,)
    x          = _np.arange(L)

    # ---- layer labels -------------------------------------------------------
    # Hidden layers: label as 1 .. L
    layer_labels = [str(i + 1) for i in range(L)]

    # ---- plot ---------------------------------------------------------------
    fig, ax = plt.subplots(figsize=figsize)

    # Individual seed traces (faint)
    if plot_individual:
        for i in range(K):
            ax.plot(x, betti_mat[i], color=color,
                    alpha=alpha_individual, linewidth=lw_individual)

    # Mean line
    ax.plot(x, mean_curve, color=color, linewidth=lw_mean,
            marker=marker, label=f'Mean (n={K})')

    # ±1 std band
    ax.fill_between(x, mean_curve - std_curve, mean_curve + std_curve,
                    color=color, alpha=0.2, linewidth=0, label='±1 std')

    # Axes
    ax.set_xticks(x)
    ax.set_xticklabels(layer_labels, rotation=0)
    ax.set_xlabel('Hidden layer', fontsize=14, labelpad=6)

    betti_str = r'$\beta_{' + str(dims[0]) + r'}$'
    for d in dims[1:]:
        betti_str += r' + $\beta_{' + str(d) + r'}$'
    ax.set_ylabel(betti_str, fontsize=14, labelpad=6)

    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    ax.tick_params(axis='both', which='major', labelsize=11)

    # y-ticks: integer steps when range is small
    ymax = int(_np.max(betti_mat))
    if ymax <= 20:
        ax.set_yticks(_np.arange(0, ymax + 2, 1))

    if title is None:
        title = (
            rf'{betti_str} — {arch_name} / {act_name},  '
            rf'$\eta={eta}$,  K={K} seeds'
        )
    ax.set_title(title, fontsize=15, pad=10)

    plt.tight_layout()

    if save:
        dim_str  = ''.join(str(d) for d in dims)
        fname    = filename if filename else f'{arch_name}_{act_name}'
        out_path = FIG_ROOT / f'betti_curves/{fname}_B{dim_str}_eta{eta}.png'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        print(f'  Saved → {out_path}')

    plt.show()
    plt.close(fig)

    return mean_curve, std_curve, betti_mat


# ── Run: one figure per (arch × activation) pair, saved automatically ────────
SAVE_BETTI_FIGS = True   # set False to only display, not save

for arch_name in ARCHS:
    for act_name in ACTIVATIONS:
        print(f'Plotting {arch_name} / {act_name} ...')
        plot_betti_curve_ann(
            arch_name       = arch_name,
            act_name        = act_name,
            eta             = PLOT_ETA,
            dims            = DIMS,
            n_seeds         = N_SEEDS,
            dir_name        = DIR_NAME,
            use_running_min = False,
            save            = SAVE_BETTI_FIGS,
        )
